In [2]:
import os
import pickle
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import numpy as np
from tqdm import tqdm

# =====================================================
# CONFIGURATION
# =====================================================
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"🚀 Using device: {DEVICE}")

TRAIN_PKL = "/home/sunkari/Stock_price_predictor/Transformer/windows_b/train_windows.pkl"
TEST_PKL = "/home/sunkari/Stock_price_predictor/Transformer/windows_b/test_windows.pkl"
MODEL_PATH = "bilstm_model.pt"

BATCH_SIZE = 64
EPOCHS = 100
LEARNING_RATE = 1e-3
HIDDEN_SIZE = 64
NUM_LAYERS = 2
GRAD_CLIP = 0.5

# =====================================================
# 1️⃣ LOAD DATA
# =====================================================
with open(TRAIN_PKL, "rb") as f:
    train_data = pickle.load(f)
with open(TEST_PKL, "rb") as f:
    test_data = pickle.load(f)

print(f"✅ Train samples: {len(train_data)}, Test samples: {len(test_data)}")

# =====================================================
# 2️⃣ CREATE DATASET
# =====================================================
class WindowDataset(Dataset):
    def __init__(self, data):
        self.X = [x for x, y in data]
        self.y = [y for x, y in data]

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return (
            torch.tensor(self.X[idx], dtype=torch.float32),
            torch.tensor(self.y[idx], dtype=torch.float32)
        )

train_loader = DataLoader(WindowDataset(train_data), batch_size=BATCH_SIZE, shuffle=True)
test_loader = DataLoader(WindowDataset(test_data), batch_size=BATCH_SIZE, shuffle=False)

# =====================================================
# 3️⃣ DEFINE BiLSTM MODEL
# =====================================================
class BiLSTMRegressor(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers):
        super().__init__()
        self.hidden_size = hidden_size
        self.num_layers = num_layers
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            bidirectional=True
        )
        self.fc = nn.Linear(hidden_size * 2, 1)  # *2 for bidirectional

    def forward(self, x):
        out, _ = self.lstm(x)  # out: (batch, seq_len, hidden*2)
        out = out[:, -1, :]    # take last time step
        out = self.fc(out)
        return out.squeeze(-1)

input_size = train_data[0][0].shape[1]  # number of features per day
model = BiLSTMRegressor(input_size, HIDDEN_SIZE, NUM_LAYERS).to(DEVICE)
print(f"✅ BiLSTM model initialized | input_size={input_size}, hidden_size={HIDDEN_SIZE}, num_layers={NUM_LAYERS}")

# =====================================================
# 4️⃣ LOSS AND OPTIMIZER
# =====================================================
criterion = nn.MSELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)

# =====================================================
# 5️⃣ TRAINING LOOP
# =====================================================
for epoch in range(EPOCHS):
    model.train()
    total_loss = 0.0
    skipped = 0

    for X_batch, y_batch in tqdm(train_loader, desc=f"Epoch {epoch+1}/{EPOCHS}"):
        X_batch, y_batch = X_batch.to(DEVICE), y_batch.to(DEVICE)
        optimizer.zero_grad()
        preds = model(X_batch)
        if torch.isnan(preds).any() or torch.isinf(preds).any():
            skipped += 1
            continue
        loss = criterion(preds, y_batch)
        if torch.isnan(loss) or torch.isinf(loss):
            skipped += 1
            continue
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / (len(train_loader) - skipped + 1e-8)
    print(f"Epoch {epoch+1:03d} | Train Loss: {avg_loss:.6f} | Skipped: {skipped}")

# =====================================================
# 6️⃣ SAVE MODEL
# =====================================================
torch.save({
    "model_state": model.state_dict(),
    "input_size": input_size,
    "hidden_size": HIDDEN_SIZE,
    "num_layers": NUM_LAYERS
}, MODEL_PATH)

print(f"✅ BiLSTM model saved as '{MODEL_PATH}'")


🚀 Using device: cuda
✅ Train samples: 7960, Test samples: 1990
✅ BiLSTM model initialized | input_size=9, hidden_size=64, num_layers=2


Epoch 1/100: 100%|██████████| 125/125 [00:01<00:00, 88.26it/s] 


Epoch 001 | Train Loss: 34486.270821 | Skipped: 98


Epoch 2/100: 100%|██████████| 125/125 [00:00<00:00, 171.32it/s]


Epoch 002 | Train Loss: 30668.051394 | Skipped: 100


...

Epoch 098 | Train Loss: 6665.889646 | Skipped: 100


Epoch 99/100: 100%|██████████| 125/125 [00:00<00:00, 183.66it/s]


Epoch 099 | Train Loss: 7002.739588 | Skipped: 103


Epoch 100/100: 100%|██████████| 125/125 [00:00<00:00, 180.77it/s]

Epoch 100 | Train Loss: 6711.649355 | Skipped: 99
✅ BiLSTM model saved as 'bilstm_model.pt'
